# Projet 4 - Finance quantique-inspirée

## Question et périmètre

Ce projet teste si un modèle de révision de croyances dont l'ordre des preuves compte peut produire des signaux différents d'une fusion classique. « Quantique » décrit ici une analogie mathématique inspirée des probabilités non commutatives; le modèle n'utilise pas d'ordinateur quantique et n'est pas une théorie financière validée.

L'univers est le top 10 des sociétés **cotées** par capitalisation boursière dans le classement CompaniesMarketCap consulté le 22 novembre 2026. SpaceX, classée dans cette page mais non cotée, est exclue; Tesla devient la dixième société cotée retenue. Le classement change avec les prix et les sources : cette liste est un instantané, pas une sélection historique sans biais.

| Rang coté | Société | Ticker Yahoo Finance |
|---:|---|---|
| 1 | Apple | AAPL |
| 2 | Microsoft | MSFT |
| 3 | Alphabet | GOOG |
| 4 | NVIDIA | NVDA |
| 5 | Amazon | AMZN |
| 6 | Saudi Aramco | 2222.SR |
| 7 | TSMC (ADR) | TSM |
| 8 | Broadcom | AVGO |
| 9 | Meta Platforms | META |
| 10 | Tesla | TSLA |

Source du classement : [CompaniesMarketCap - Largest Companies by Market Cap](https://companiesmarketcap.com/). La couverture historique commence lorsque les dix séries ont toutes des cours; les jours fériés propres à chaque marché sont alignés en conservant le dernier cours connu.

## Modèle et protocole de test

Chaque titre possède une matrice de croyance 2×2, positive et de trace 1, pour les états « hausse » et « baisse ». Deux probabilités d'évidence sont calculées sans données futures : rendement de momentum sur 126 séances, normalisé par la volatilité, et distance au niveau de tendance sur 200 séances. Les scores passent par une fonction logistique.

Une preuve est représentée par un effet positif `E = R diag(p, 1-p) Rᵀ`; la révision de Lüders est `ρ' = √E ρ √E / Tr(√E ρ √E)`. Le momentum est exprimé dans la base canonique et la tendance dans une base tournée de 45°, donc les opérateurs peuvent ne pas commuter. Une mémoire de 0,95 mélange l'état précédent avec un état neutre avant la révision quotidienne.

Les deux séquences momentum→tendance et tendance→momentum sont évaluées. Le modèle quantique n'achète qu'en accord haussier des deux séquences (probabilités ≥ 0,55), vend en accord baissier (≤ 0,45) et conserve sinon. Le témoin classique fait la moyenne des deux probabilités avant d'appliquer les mêmes seuils. Les deux stratégies sont long-only, équipondérées entre positions ouvertes, décalées d'une séance, et déduisent 0,1 % du turnover. Le benchmark est un panier équipondéré achat-conservation.

Limites : l'univers sélectionné aujourd'hui crée un biais de survivance et d'anticipation dans le choix des titres; le classement par capitalisation n'est pas lui-même historique. Le backtest est exploratoire, sensible aux seuils, à l'angle et à la mémoire, et ne démontre ni avantage statistique ni performance future. Les sorties ACHAT/VENTE/ATTENDRE sont des signaux de démonstration, pas des recommandations.

In [ ]:
from pathlib import Path
import sys

racine = Path.cwd().resolve()
while not (racine / "src" / "quantum_belief_strategy.py").is_file() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine / "src"))

import pandas as pd
from metrics import metriques
from quantum_belief_strategy import UNIVERS_TITRES, backtest_revision_quantique

TICKERS = [ticker for ticker, _ in UNIVERS_TITRES]
resultat = backtest_revision_quantique(
    TICKERS,
    debut="2018-01-01",
    momentum_window=126,
    trend_window=200,
    seuil_achat=0.55,
    seuil_vente=0.45,
    angle_degres=45,
    memoire=0.95,
    cout=0.001,
)
print(f"Backtest : {resultat.index[0].date()} — {resultat.index[-1].date()} ({len(resultat):,} séances)")

In [ ]:
comparaison = pd.DataFrame(
    {
        "Révision non commutative": metriques(resultat["strategie_quantique"]),
        "Fusion classique": metriques(resultat["strategie_classique"]),
        "Achat-conservation équipondéré": metriques(resultat["benchmark_equipondere"]),
    }
)
display(comparaison)

decisions = pd.DataFrame(
    [
        {
            "Société": nom,
            "Ticker": ticker,
            "Décision la plus récente": resultat[f"decision_{ticker}"].iloc[-1],
            "Position appliquée": resultat[f"position_quantique_{ticker}"].iloc[-1],
            "Écart entre ordres": resultat.attrs["croyances"][(ticker, "ecart_ordre")].iloc[-1],
        }
        for ticker, nom in UNIVERS_TITRES
    ]
)
display(decisions)

chemin_csv = racine / "backtest" / "projet4_finance_quantique.csv"
chemin_csv.parent.mkdir(parents=True, exist_ok=True)
resultat.to_csv(chemin_csv)
print(f"Séries et décisions enregistrées dans {chemin_csv}")

In [ ]:
import matplotlib.pyplot as plt

ax = (
    (1 + resultat["benchmark_equipondere"]).cumprod() * 100
).plot(figsize=(14, 7), color="black", linewidth=2, label="Achat-conservation équipondéré")
((1 + resultat["strategie_classique"]).cumprod() * 100).plot(
    ax=ax, label="Fusion classique", alpha=0.85
)
((1 + resultat["strategie_quantique"]).cumprod() * 100).plot(
    ax=ax, label="Révision non commutative", alpha=0.85
)
ax.set_title("Projet 4 - Comparaison des stratégies")
ax.set_ylabel("Valeur (base 100)")
ax.grid(True)
ax.legend()
plt.tight_layout()

## Repères théoriques

- [Logic of Belief Revision, Stanford Encyclopedia of Philosophy](https://plato.stanford.edu/entries/logic-belief-revision/) : opérations formelles de révision, rationalité et variantes non prioritaires.
- Bruza, Wang & Busemeyer (2015), [Quantum cognition: a new theoretical approach to psychology](https://doi.org/10.1016/j.tics.2015.05.001) : l'utilisation de structures de probabilité quantique comme modèles mathématiques de contextes et d'effets d'ordre en cognition; cela ne constitue pas une preuve d'efficacité financière.